# 6.1 Audio Encoder — Design Experiment (Step 1 of 2)

**Google Colab Pro edition (L4 / A100 ready)** — satu notebook untuk semua `MEMBER` (1–4). Jalankan sel **Setup** dan **Cache**
sebelum sel eksperimen. Detail langkah ada di `notebooks/colab/README.md`.

**Tujuan:** memilih **audio encoder terbaik** untuk *text-guided automatic gain staging* pada
MUSDB18-HQ, dengan protokol yang sama untuk semua encoder.

---

## Kontrak fairness

| Aspek | Nilai |
| --- | --- |
| Text encoder | `indobenchmark/indobert-base-p1`, fine-tuned |
| Text hyperparameter | Baseline `dropout_text=0.15`; text tuning OFAT opsional |
| Audio hyperparameter | Ditune **per member** dengan OFAT holdout |
| Head / arsitektur | M3 Structured Two-Branch |
| Loss | `MSE(g) + 0.1*MSE(L) + 0.1*MSE(r)` sesuai mode |
| Label | `compute_gain_targets` di `wadidaw/data/features.py` |
| Holdout | 80 lagu train / 20 lagu validation / 50 lagu test |
| Validation | 20 lagu, dipakai early stopping |
| Test | 50 lagu, tidak dipakai sampai evaluasi final |
| Optimizer | AdamW |
| Seeds | tuning `42`; final `42, 123, 2026` |
| Metrik | `evaluate_model` (MAE/RMSE/band/success/adherence/per-stem) |

**Yang berubah hanya audio encoder:** `cnn_scratch`, `ast`, `mert`, `beats` — semuanya trainable.

## Hyperparameter yang ditune

Audio per member:
- `lr_audio` / `lr_scratch`
- `wd_audio`
- `dropout_audio`

Text global:
- `lr_text`
- `dropout_text`
- `text_hidden`

Text branch tidak ditune per member agar tetap menjadi variabel kontrol.


## Cara menjalankan di Google Colab Pro (L4 / A100)

### Struktur Google Drive yang dipakai
Default path diarahkan ke folder proyek di Google Drive:
- Dataset MUSDB18-HQ (`train/` & `test/`): `/content/drive/MyDrive/ml_dabd_nlp/ml/1.Dataset/dataset`
- Cache log-mel (593 MB): `/content/drive/MyDrive/ml_dabd_nlp/ml/1.Dataset/cache/logmel`
- Cache waveform 16 kHz (±1,5 GB, dibuat otomatis di Cell 2 lalu di-backup ke sini): `/content/drive/MyDrive/ml_dabd_nlp/ml/1.Dataset/cache/waveform16k`
- Output persisten (`member1/` … `member4/`, `text/`, `final/`): `/content/drive/MyDrive/ml_dabd_nlp/ml/2.Dokumentasi/m_all`

> Seluruh training berjalan di SSD lokal VM `/content/wadidaw-studio` (cache di-copy dari Drive ke SSD lokal
> + preload RAM bila RAM ≥ 24 GB) dan memakai TF32 + `bfloat16` AMP pada L4/A100 agar GPU bekerja maksimal
> sambil menjaga `effective_batch = 16` tetap sama persis.

### Cukup set `MEMBER` (contoh `1` untuk `cnn_scratch`)

Di **Cell 1**:

```python
MEMBER = 1  # 1 / 2 / 3 / 4
AUTO_RUN = True
```

Lalu jalankan **Setup → Cache → Cell 1 → Cell 2–8 → Cell 9 → Cell 10–13** (atau *Run all*).

### Phase 1 — paralel semua member

Otomatis:

```text
audio tuning bimodal
audio-only ablation
```

`text_only` hanya dicoba oleh member 2, dan hanya jika `text_config.json` sudah ada.

### Phase 2 — setelah semua hasil audio masuk

Setelah semua member push hasil ke repo:
Jalankan lagi notebook di member pemenang (`AUTO_PICK_WINNER = True`) untuk final model 3 seed. Text tuning OFAT opsional.

> **Hemat CCU:** Setelah output dipastikan tersinkron ke Google Drive (`2.Dokumentasi/m3`), klik **Runtime → Disconnect and delete runtime**.


In [1]:
# ----------------------------------------------------------------------------- Colab setup & Google Drive
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

DRIVE_ROOT = Path("/content/drive/MyDrive")
if IN_COLAB and not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

# --- Konfigurasi path Google Drive & repo ------------------------------------
REPO_URL = "https://github.com/savitar393/wadidaw-studio.git"
BRANCH = "feat/audio-encoder-design-experiment-notebook"
REPO_DIR = Path("/content/wadidaw-studio" if IN_COLAB else Path.cwd())

DRIVE_PROJECT_DIR = DRIVE_ROOT / "ml_dabd_nlp" / "ml"
DRIVE_DATASET_HINT = DRIVE_PROJECT_DIR / "1.Dataset" / "dataset"
DRIVE_CACHE_ROOT = DRIVE_PROJECT_DIR / "1.Dataset" / "cache"
DRIVE_OUTPUT_DIR = DRIVE_PROJECT_DIR / "2.Dokumentasi" / "m4"


def run(cmd: str, check: bool = True) -> None:
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=check)


if IN_COLAB:
    if not (REPO_DIR / "wadidaw").is_dir():
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR, ignore_errors=True)
        run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    else:
        print("repo sudah ada:", REPO_DIR)
        run(f"git -C '{REPO_DIR}' pull --ff-only", check=False)
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "wadidaw").is_dir():
            REPO_DIR = candidate
            break

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("cwd:", Path.cwd())

# Dependensi untuk IndoBERT / AST / MERT / Optuna.
run(
    f"{sys.executable} -m pip install -q 'transformers>=4.40' soundfile scipy 'optuna>=3.6'",
    check=False,
)
run(f"{sys.executable} -m pip install -q --no-deps nnAudio", check=False)


def looks_like_musdb(path: Path) -> bool:
    try:
        return (
            (path / "train").is_dir()
            and (path / "test").is_dir()
            and next((path / "train").glob("*/mixture.wav"), None) is not None
        )
    except OSError:
        return False


def find_musdb_root() -> Path | None:
    if looks_like_musdb(DRIVE_DATASET_HINT):
        return DRIVE_DATASET_HINT
    search_roots = [
        DRIVE_PROJECT_DIR / "1.Dataset",
        DRIVE_PROJECT_DIR,
        DRIVE_ROOT,
        REPO_DIR / "data" / "raw",
    ]
    for base in search_roots:
        if not base.exists():
            continue
        for pat in ("*", "*/*", "*/*/*"):
            for candidate in base.glob(pat):
                if looks_like_musdb(candidate):
                    return candidate
    return None


MUSDB_ROOT_COLAB = find_musdb_root()
if MUSDB_ROOT_COLAB is None:
    zips = sorted((DRIVE_PROJECT_DIR / "1.Dataset").glob("**/*.zip")) if (DRIVE_PROJECT_DIR / "1.Dataset").exists() else []
    if zips:
        dest = Path("/content/tmp/musdb18hq")
        dest.mkdir(parents=True, exist_ok=True)
        run(f"unzip -q -o '{zips[0]}' -d '{dest}'")
        MUSDB_ROOT_COLAB = next(
            (p for p in [dest, *dest.glob("*"), *dest.glob("*/*")] if looks_like_musdb(p)),
            None,
        )

if MUSDB_ROOT_COLAB is None:
    raise AssertionError(
        f"MUSDB18-HQ tidak ditemukan. Pastikan {DRIVE_DATASET_HINT} berisi folder train/ dan test/."
    )

print("MUSDB18-HQ:", MUSDB_ROOT_COLAB)

# Symlink agar modul yang memakai MUSDB_ROOT bawaan repo tetap menunjuk ke dataset Drive.
link = REPO_DIR / "data" / "raw" / "musdb18hq"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.exists():
    print("path MUSDB lokal sudah ada:", link)
else:
    link.symlink_to(MUSDB_ROOT_COLAB, target_is_directory=True)
    print("symlink:", link, "->", MUSDB_ROOT_COLAB)

import wadidaw.data.musdb as _musdb_module

_musdb_module.MUSDB_ROOT = MUSDB_ROOT_COLAB

split_manifest = json.loads((REPO_DIR / "data" / "splits" / "musdb18hq_v1.json").read_text())
songs = [member for split in split_manifest["splits"].values() for member in split]
missing_songs = [
    member["song"]
    for member in songs
    if not (MUSDB_ROOT_COLAB / str(member["relative_dir"])).is_dir()
]
print(f"split songs: {len(songs)} | folder tidak ditemukan: {len(missing_songs)}")
if missing_songs:
    print("contoh hilang:", missing_songs[:5])

# Restore hasil sebelumnya dari Google Drive (jika ada) untuk mendukung resume antar sesi.
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT_DIR = REPO_DIR / "outputs" / "encoder_selection"
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
for sub in ("member1", "member2", "member3", "member4", "text", "final", "folds"):
    src_sub = DRIVE_OUTPUT_DIR / sub
    dst_sub = LOCAL_OUT_DIR / sub
    if src_sub.is_dir() and any(src_sub.iterdir()):
        shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)
        print("restored from Drive:", dst_sub)

import torch

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    props = torch.cuda.get_device_properties(0)
    print(
        f"torch: {torch.__version__} | cuda: True | gpu: {props.name} "
        f"({props.total_memory / 1e9:.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}"
    )
else:
    print("torch:", torch.__version__, "| cuda: False")


Mounted at /content/drive
+ git clone --depth 1 --branch feat/audio-encoder-design-experiment-notebook https://github.com/savitar393/wadidaw-studio.git /content/wadidaw-studio
cwd: /content/wadidaw-studio
+ /usr/bin/python3 -m pip install -q 'transformers>=4.40' soundfile scipy 'optuna>=3.6'
+ /usr/bin/python3 -m pip install -q --no-deps nnAudio
MUSDB18-HQ: /content/drive/MyDrive/dataset
symlink: /content/wadidaw-studio/data/raw/musdb18hq -> /content/drive/MyDrive/dataset
split songs: 150 | folder tidak ditemukan: 0
torch: 2.14.1+cu130 | cuda: True | gpu: NVIDIA L4 (23.7 GB) | bf16: True


In [2]:
# ----------------------------------------------------------------------------- Colab cache (Drive <-> SSD lokal)
from wadidaw.data.dataset import SAMPLES_ROOT

LOGMEL_CACHE = REPO_DIR / "data" / "cache" / "logmel"
WAVEFORM_CACHE_DIR = REPO_DIR / "data" / "cache" / "waveform16k"
DRIVE_LOGMEL_CACHE = DRIVE_CACHE_ROOT / "logmel"
DRIVE_WAVEFORM_CACHE = DRIVE_CACHE_ROOT / "waveform16k"

BUILD_LOGMEL_IF_MISSING = True   # False -> gagal cepat kalau cache belum ada di Drive


def has_npy_cache(path: Path) -> bool:
    try:
        return (path / "train").is_dir() and next((path / "train").glob("*.npy"), None) is not None
    except OSError:
        return False


def count_npy(path: Path) -> int:
    return sum(1 for _ in path.rglob("*.npy")) if path.exists() else 0


def find_drive_cache(name: str) -> Path | None:
    direct = DRIVE_CACHE_ROOT / name
    if has_npy_cache(direct):
        return direct
    for base in (DRIVE_PROJECT_DIR / "1.Dataset", DRIVE_PROJECT_DIR):
        if not base.exists():
            continue
        for candidate in base.glob(f"**/{name}"):
            if has_npy_cache(candidate):
                return candidate
    return None


def copy_cache_to_local(source: Path, target: Path, label: str) -> None:
    if has_npy_cache(target) and count_npy(target) >= count_npy(source):
        print(f"{label} cache sudah ada di SSD lokal: {target} ({count_npy(target)} file)")
        return
    if target.is_symlink():
        target.unlink()
    target.mkdir(parents=True, exist_ok=True)
    print(f"Menyalin {label} cache dari Drive ({source}) ke SSD lokal ({target})...")
    shutil.copytree(source, target, dirs_exist_ok=True)
    print(f"Selesai copy {label}: {count_npy(target)} file")


drive_logmel = find_drive_cache("logmel")
if drive_logmel is not None:
    copy_cache_to_local(drive_logmel, LOGMEL_CACHE, "log-mel")
elif has_npy_cache(LOGMEL_CACHE):
    print("log-mel cache sudah ada di SSD lokal:", LOGMEL_CACHE)
elif BUILD_LOGMEL_IF_MISSING:
    print("log-mel cache belum ada -> membangun dari WAV (30-60 menit) ...")
    run(
        f"{sys.executable} -m scripts.build_dataset "
        f"--musdb-root '{MUSDB_ROOT_COLAB}' "
        f"--manifest '{REPO_DIR / 'data' / 'splits' / 'musdb18hq_v1.json'}' "
        f"--cache-root '{LOGMEL_CACHE}' "
        "--samples-root /content/tmp/prompts_colab "
        "--workers 4"
    )
    if IN_COLAB and DRIVE_ROOT.exists():
        DRIVE_LOGMEL_CACHE.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(LOGMEL_CACHE, DRIVE_LOGMEL_CACHE, dirs_exist_ok=True)
        print("Backed up log-mel cache -> Drive:", DRIVE_LOGMEL_CACHE)
else:
    raise FileNotFoundError(
        f"log-mel cache tidak ditemukan di {DRIVE_LOGMEL_CACHE} maupun {LOGMEL_CACHE}."
    )

drive_waveform = find_drive_cache("waveform16k")
if drive_waveform is not None and count_npy(drive_waveform) >= 1500:
    copy_cache_to_local(drive_waveform, WAVEFORM_CACHE_DIR, "waveform16k")
elif drive_waveform is not None:
    print(f"waveform16k di Drive baru {count_npy(drive_waveform)}/1500 file; copy parsial lalu dilengkapi di Cell 2.")
    copy_cache_to_local(drive_waveform, WAVEFORM_CACHE_DIR, "waveform16k")

missing_audio = []
for manifest_name in (
    "train_samples.json",
    "validation_samples.json",
    "test_human_samples.json",
    "test_synthetic_samples.json",
):
    manifest_path = SAMPLES_ROOT / manifest_name
    if not manifest_path.is_file():
        continue
    for record in json.loads(manifest_path.read_text()):
        if not (REPO_DIR / str(record["audio_path"])).is_file():
            missing_audio.append(record["audio_path"])

print(f"manifest audio files missing: {len(missing_audio)}")
if missing_audio:
    print("contoh:", missing_audio[:3])
    raise FileNotFoundError("cache log-mel tidak lengkap; cek folder 1.Dataset/cache/logmel di Drive")
print("cache log-mel siap di SSD lokal:", LOGMEL_CACHE)


log-mel cache belum ada -> membangun dari WAV (30-60 menit) ...
+ /usr/bin/python3 -m scripts.build_dataset --musdb-root '/content/drive/MyDrive/dataset' --manifest '/content/wadidaw-studio/data/splits/musdb18hq_v1.json' --cache-root '/content/wadidaw-studio/data/cache/logmel' --samples-root /content/tmp/prompts_colab --workers 4
Backed up log-mel cache -> Drive: /content/drive/MyDrive/ml_dabd_nlp/ml/1.Dataset/cache/logmel
manifest audio files missing: 0
cache log-mel siap di SSD lokal: /content/wadidaw-studio/data/cache/logmel


In [3]:
# ----------------------------------------------------------------------------- Cell 1: bootstrap + config
import copy
import json
import os
import random
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "wadidaw").is_dir():
            return candidate
    return start


def system_ram_gb() -> float:
    try:
        pages = os.sysconf("SC_PHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        return (pages * page_size) / 1e9
    except (ValueError, AttributeError):
        return 0.0


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from wadidaw.data.dataset import SAMPLES_ROOT, MusdbMultimodalDataset
from wadidaw.data.features import ACTIVE_THRESHOLD_DB, resample_to_16k_mono
from wadidaw.data.musdb import MUSDB_ROOT
from wadidaw.data.prompts import STEM_NAMES
from wadidaw.models.structured import compute_multimodal_loss
from wadidaw.training.evaluate import evaluate_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WAVEFORM_CACHE = ROOT / "data" / "cache" / "waveform16k"
OUT_DIR = ROOT / "outputs" / "encoder_selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Path persisten Google Drive (disamakan dengan sel Setup)
if "DRIVE_OUTPUT_DIR" not in globals():
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ml_dabd_nlp/ml/2.Dokumentasi/m3")
if "DRIVE_CACHE_ROOT" not in globals():
    DRIVE_CACHE_ROOT = Path("/content/drive/MyDrive/ml_dabd_nlp/ml/1.Dataset/cache")

CONFIG = {
    "train_manifest": str(SAMPLES_ROOT / "train_samples.json"),
    "val_manifest": str(SAMPLES_ROOT / "validation_samples.json"),
    "test_human_manifest": str(SAMPLES_ROOT / "test_human_samples.json"),
    "test_synthetic_manifest": str(SAMPLES_ROOT / "test_synthetic_samples.json"),
    "text_model": "indobenchmark/indobert-base-p1",
    "d_model": 128,
    "text_hidden": 128,
    "dropout_text": 0.15,
    "dropout_audio": 0.10,
    "batch_size": 4,
    "grad_accum_steps": 4,
    "epochs": 7,
    "patience": 3,
    "lr_head": 1e-3,
    "lr_audio": 1e-4,
    "lr_scratch": 1e-3,
    "lr_text": 1e-4,
    "wd_audio": 1e-4,
    "wd_text": 1e-4,
    "wd_head": 1e-4,
    "eta_min": 1e-6,
    "grad_clip": 1.0,
    "seeds": [42, 123, 2026],
    "use_amp": True,
    "encoders": ["cnn_scratch", "ast", "mert", "beats"],
    "modes": ["bimodal", "audio_only", "text_only"],
    "waveform_rate": 16000,
    "n_folds": 5,
    "cv_seed": 2026,
    "tuning_seed": 42,
    "n_trials": 1,
    "audio_only_seeds": [42],
}
# Auxiliary-loss weights per ablation mode (lambda_loudness, lambda_balance).
AUX_LAMBDA = {"bimodal": (0.1, 0.1), "audio_only": (0.1, 0.0), "text_only": (0.0, 0.1)}

# --- Pembagian encoder ke 4 anggota -----------------------------------------
MEMBER = int(os.environ.get("WADIDAW_MEMBER", "4"))
SHARDS = {
    1: {"encoders": ["cnn_scratch"], "modes": ["bimodal", "audio_only"]},                 # iGPU Arc / CPU, 16 GB
    2: {"encoders": ["ast"],          "modes": ["bimodal", "audio_only", "text_only"]},   # RTX 3050 4 GB, 24 GB
    3: {"encoders": ["mert"],         "modes": ["bimodal", "audio_only"]},                 # RTX 4050 6 GB, 32 GB
    4: {"encoders": ["beats"],        "modes": ["bimodal", "audio_only"]},                 # RTX 5050 8 GB, 16 GB
}
effective_batch = 16
assert MEMBER in SHARDS, f"MEMBER harus 1..4, bukan {MEMBER}"

# Maksimalkan GPU L4 (24 GB) / A100 (40+ GB) sambil menjaga effective_batch == 16.
# cnn_scratch memakai BatchNorm1d -> tetap pakai 4x4; encoder transformer (MERT/AST/BEATs) pakai LayerNorm.
AUTO_GPU_BATCH = True
if AUTO_GPU_BATCH and torch.cuda.is_available() and "cnn_scratch" not in SHARDS[MEMBER]["encoders"]:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 36.0:      # A100 40/80 GB
        CONFIG["batch_size"] = 16
        CONFIG["grad_accum_steps"] = 1
    elif vram_gb >= 20.0:    # L4 24 GB
        CONFIG["batch_size"] = 8
        CONFIG["grad_accum_steps"] = 2

assert CONFIG["batch_size"] * CONFIG["grad_accum_steps"] == effective_batch, "effective batch harus 16"

RUN_DIR = OUT_DIR / f"member{MEMBER}"
FOLD_FILE = ROOT / "data" / "splits" / "cv5_train_songs.json"
FOLD_DIR = OUT_DIR / "folds"
TEXT_DIR = OUT_DIR / "text"
FINAL_DIR = OUT_DIR / "final"
CV_DIR = RUN_DIR / "cv"
STUDY_DIR = RUN_DIR / "optuna"
for _d in (OUT_DIR, RUN_DIR, FOLD_DIR, TEXT_DIR, FINAL_DIR, CV_DIR, STUDY_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Stage flags -------------------------------------------------------------
AUTO_RUN = True
AUTO_PICK_WINNER = True
FORCE_RERUN = False

# Manual fallback. Diabaikan kalau AUTO_RUN=True.
RUN_AUDIO_TUNING = False
RUN_TEXT_TUNING = False
RUN_FINAL = False
RUN_AUDIO_ONLY = False
RUN_TEXT_ONLY = False

TEXT_TUNING_ENCODER = None
TEXT_TUNING_MEMBER = None
FINAL_ENCODER = None

# --- Search spaces -----------------------------------------------------------
AUDIO_SEARCH_SPACE = {
    "pretrained": {
        "lr_audio": [5e-5, 1e-4, 2e-4],
        "wd_audio": [1e-4, 1e-3],
        "dropout_audio": [0.0, 0.1, 0.2],
    },
    "scratch": {
        "lr_scratch": [5e-4, 1e-3, 2e-3],
        "wd_audio": [1e-4, 1e-3],
        "dropout_audio": [0.0, 0.1, 0.2],
    },
}
TEXT_SEARCH_SPACE = {
    "lr_text": [5e-5, 1e-4, 2e-4],
    "dropout_text": [0.10, 0.15, 0.20],
    "text_hidden": [64, 128, 256],
}

SAVE_FINAL_CHECKPOINT = True
MAX_TRIALS_PER_SESSION = 1   # satu trial per sesi; sesuaikan bila ingin >1 trial
TIME_BUDGET_MIN = 660
NUM_WORKERS = min(4, os.cpu_count() or 2) if torch.cuda.is_available() else 0
PIN_MEMORY = bool(torch.cuda.is_available())
PRELOAD_CACHE_IN_RAM = system_ram_gb() >= 24.0   # muat ~2.1 GB cache ke RAM di L4/A100 VM
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

# BEATs + AMP FP16 tidak stabil di T4; tetap matikan AMP untuk BEATs saat FP16.
AMP_DISABLED_ENCODERS = {"beats"} if AMP_DTYPE == torch.float16 else set()
DEBUG_NUMERICS = True

print("device:", DEVICE, "| amp_dtype:", AMP_DTYPE, "| num_workers:", NUM_WORKERS, "| preload_ram:", PRELOAD_CACHE_IN_RAM)
print("repo root:", ROOT)
print("waveform cache:", WAVEFORM_CACHE)
print("drive output:", DRIVE_OUTPUT_DIR)
print(
    f"member {MEMBER}: encoders={SHARDS[MEMBER]['encoders']} modes={SHARDS[MEMBER]['modes']} "
    f"(batch={CONFIG['batch_size']} x accum={CONFIG['grad_accum_steps']} = effective {effective_batch})"
)
print(
    f"auto={AUTO_RUN} pick_winner={AUTO_PICK_WINNER} force={FORCE_RERUN} | "
    f"manual: audio={RUN_AUDIO_TUNING} text={RUN_TEXT_TUNING} final={RUN_FINAL} "
    f"audio_only={RUN_AUDIO_ONLY} text_only={RUN_TEXT_ONLY}"
)
print(f"tuning: n_trials={CONFIG['n_trials']} tuning_seed={CONFIG['tuning_seed']} final_seeds={CONFIG['seeds']}")


device: cuda | amp_dtype: torch.bfloat16 | num_workers: 4 | preload_ram: True
repo root: /content/wadidaw-studio
waveform cache: /content/wadidaw-studio/data/cache/waveform16k
drive output: /content/drive/MyDrive/ml_dabd_nlp/ml/2.Dokumentasi/m4
member 4: encoders=['beats'] modes=['bimodal', 'audio_only'] (batch=8 x accum=2 = effective 16)
auto=True pick_winner=True force=False | manual: audio=False text=False final=False audio_only=False text_only=False
tuning: n_trials=1 tuning_seed=42 final_seeds=[42, 123, 2026]


## Pembagian kerja 4 anggota

| Anggota | Encoder | Hyperparameter audio yang ditune |
| --- | --- | --- |
| 1 | `cnn_scratch` | `lr_scratch`, `wd_audio`, `dropout_audio` |
| 2 | `ast` | `lr_audio`, `wd_audio`, `dropout_audio` |
| 3 | `mert` | `lr_audio`, `wd_audio`, `dropout_audio` |
| 4 | `beats` | `lr_audio`, `wd_audio`, `dropout_audio` |

Text hyperparameter ditune **global sekali** oleh member pemilik encoder pemenang:
`lr_text`, `dropout_text`, `text_hidden`.

Batch dikunci `4 × 4 = effective 16` di semua mesin.

### Alur otomatis

1. Semua member set `MEMBER` dan `AUTO_RUN=True`, lalu jalankan notebook.
2. Phase 1 otomatis: audio tuning + audio-only ablation.
3. Setelah semua hasil audio masuk, member pemenang menjalankan notebook lagi.
4. `AUTO_PICK_WINNER=True` menentukan winner dari CV MAE.
5. Winner otomatis menjalankan text tuning + final model.
6. Member 2 menjalankan text-only ablation setelah `text_config.json` tersedia.

Stage yang sudah selesai otomatis di-skip. Gunakan `FORCE_RERUN=True` hanya jika
benar-benar ingin mengulang.

> Anggota 2 (4 GB): AST fine-tune `batch 4` bisa OOM. Fallback yang tetap adil adalah
> turunkan `batch_size` dan naikkan `grad_accum_steps` agar hasil kali tetap 16 (mis. `2×8`).


## 1. Cache waveform 16 kHz

AST/BEATs memakai 16 kHz, MERT-v1 aslinya 24 kHz. Kita cache **satu** waveform 16 kHz mono
(reuse `resample_to_16k_mono`, konsisten dengan cache log-mel) dan memberi MERT input 16 kHz
sebagai *documented domain shift*; upgrade ke 24 kHz jika MERT menang.

Setiap excerpt di-cache sebagai `(4, 128000)` float16 (stem urut `STEM_NAMES`), **raw tanpa g0** —
perturbasi g0 diterapkan saat load, sama seperti cache log-mel.

In [4]:
# ----------------------------------------------------------------------------- Cell 2: waveform cache (idempotent + backup ke Drive)
from concurrent.futures import ThreadPoolExecutor


def excerpt_id_from_audio_path(audio_path: str) -> tuple[str, str]:
    p = Path(audio_path)
    return p.stem, p.parent.name  # (excerpt_id, split_tag)


def waveform_cache_path(audio_path: str) -> Path:
    ex_id, split_tag = excerpt_id_from_audio_path(audio_path)
    return WAVEFORM_CACHE / split_tag / f"{ex_id}.npy"


def musdb_subset_for_split(split_tag: str) -> str:
    return "train" if split_tag in ("train", "validation") else "test"


def _build_one_waveform(args: tuple[str, dict, bool]) -> int:
    audio_path, record, overwrite = args
    npy_path = waveform_cache_path(audio_path)
    if npy_path.exists() and not overwrite:
        return 0
    _ex_id, split_tag = excerpt_id_from_audio_path(audio_path)
    song_dir = MUSDB_ROOT / musdb_subset_for_split(split_tag) / str(record["song"])
    start = int(round(float(record["excerpt_start_sec"]) * 44100))
    frames = int(round(float(record["duration_sec"]) * 44100))
    stems = []
    for stem in STEM_NAMES:
        audio, sr = sf.read(song_dir / f"{stem}.wav", start=start, frames=frames, dtype="float32")
        stems.append(resample_to_16k_mono(audio, orig_sr=sr))
    npy_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(npy_path, np.stack(stems, axis=0).astype(np.float16))
    return 1


def build_waveform_cache(manifests, overwrite: bool = False, workers: int = 8) -> int:
    """Cache 16 kHz mono (4, 128000) float16 waveforms for every excerpt in the manifests."""
    unique: dict[str, dict] = {}
    for manifest in manifests:
        if not Path(manifest).is_file():
            continue
        for record in MusdbMultimodalDataset(manifest).samples:
            unique.setdefault(str(record["audio_path"]), record)

    tasks = [(ap, rec, overwrite) for ap, rec in unique.items()]
    built = 0
    if workers > 1:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            for done in pool.map(_build_one_waveform, tasks):
                built += done
                if built and built % 250 == 0:
                    print(f"  waveform cache progress: +{built} built...")
    else:
        for t in tasks:
            built += _build_one_waveform(t)
    print(f"waveform cache: built {built}, total excerpts {len(unique)}")
    return built


CACHE_MANIFESTS = [CONFIG[k] for k in ("train_manifest", "val_manifest", "test_human_manifest", "test_synthetic_manifest")]
existing_wf = sum(1 for _ in WAVEFORM_CACHE.rglob("*.npy")) if WAVEFORM_CACHE.exists() else 0
if existing_wf < 1500:
    newly_built = build_waveform_cache(CACHE_MANIFESTS)
    drive_wf = DRIVE_CACHE_ROOT / "waveform16k"
    if (newly_built > 0 or not drive_wf.exists()) and Path("/content/drive/MyDrive").exists():
        drive_wf.parent.mkdir(parents=True, exist_ok=True)
        print(f"Menyalin waveform16k cache ke Google Drive ({drive_wf})...")
        shutil.copytree(WAVEFORM_CACHE, drive_wf, dirs_exist_ok=True)
        print("Backup waveform16k ke Drive selesai.")
else:
    print(f"waveform cache already present in local SSD: {WAVEFORM_CACHE} ({existing_wf} files)")


  waveform cache progress: +250 built...
  waveform cache progress: +500 built...
  waveform cache progress: +750 built...
  waveform cache progress: +1000 built...
  waveform cache progress: +1250 built...
  waveform cache progress: +1500 built...
waveform cache: built 1500, total excerpts 1500
Menyalin waveform16k cache ke Google Drive (/content/drive/MyDrive/ml_dabd_nlp/ml/1.Dataset/cache/waveform16k)...
Backup waveform16k ke Drive selesai.


## 2. Dataset

`EncoderDataset` membungkus `MusdbMultimodalDataset` (log-mel `(4, 64, 800)` + label) dan menambahkan
`waveform` `(4, 128000)` dengan perturbasi `g0` yang sama. `cache_in_memory=False` untuk menekan RAM.

In [5]:
# ----------------------------------------------------------------------------- Cell 3: dataset


class EncoderDataset(Dataset):
    """MusdbMultimodalDataset + raw 16 kHz waveform (same g0 perturbation as the log-mel input)."""

    def __init__(self, manifest, root_dir=ROOT, cache_in_memory: bool | None = None):
        self.root_dir = Path(root_dir)
        if cache_in_memory is None:
            cache_in_memory = bool(PRELOAD_CACHE_IN_RAM)
        self.cache_in_memory = cache_in_memory
        self.base = MusdbMultimodalDataset(manifest, root_dir=root_dir, cache_in_memory=cache_in_memory)
        self._wf_cache: dict[str, np.ndarray] = {}
        if self.cache_in_memory:
            # Preload semua unique excerpt sebelum DataLoader fork worker -> hemat disk I/O & share memory (COW).
            for idx, record in enumerate(self.base.samples):
                _ = self.base[idx]  # isi _npy_cache
                ap = str(record["audio_path"])
                if ap not in self._wf_cache:
                    wf_path = waveform_cache_path(ap)
                    if not wf_path.is_file():
                        raise FileNotFoundError(f"Missing waveform cache {wf_path}. Run Cell 2 first.")
                    self._wf_cache[ap] = np.load(wf_path)

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, index: int) -> dict:
        item = self.base[index]
        record = self.base.samples[index]
        ap = str(record["audio_path"])
        if ap in self._wf_cache:
            waveform = self._wf_cache[ap].astype(np.float32)
        else:
            wf_path = waveform_cache_path(ap)
            if not wf_path.is_file():
                raise FileNotFoundError(f"Missing waveform cache {wf_path}. Run Cell 2 first.")
            waveform = np.load(wf_path).astype(np.float32)
        g0 = np.asarray(record["g0_db"], dtype=np.float32)
        mask = np.asarray(record.get("active_mask", [True] * 4), dtype=bool)
        eff = np.where(mask, g0, 0.0).astype(np.float32)
        item["waveform"] = torch.from_numpy(waveform * np.power(10.0, eff / 20.0)[:, None])
        return item


_ds = EncoderDataset(CONFIG["val_manifest"], cache_in_memory=False)
_s = _ds[0]
print("keys:", [k for k in _s if k in ("audio", "waveform", "gains_db", "loudness_db", "balance_target", "active_mask")])
print("audio", tuple(_s["audio"].shape), "| waveform", tuple(_s["waveform"].shape), "| gains", tuple(_s["gains_db"].shape))


keys: ['audio', 'loudness_db', 'balance_target', 'gains_db', 'active_mask', 'waveform']
audio (4, 64, 800) | waveform (4, 128000) | gains (4,)


## 2b. Holdout split (tanpa CV)

5-fold CV **dihapus**. Tuning menggunakan split tetap:

```text
train      = 80 lagu
validation = 20 lagu
test       = 50 lagu
```

Validation dan test memakai prompt template yang berbeda dari train. Tidak ada fold CV.


In [ ]:
# ----------------------------------------------------------------------------- Cell 3b: holdout (CV removed)
print("5-fold CV disabled; using fixed 80/20/50 holdout.")
print("train:", CONFIG["train_manifest"])
print("validation:", CONFIG["val_manifest"])
print("test_human:", CONFIG["test_human_manifest"])
print("test_synthetic:", CONFIG["test_synthetic_manifest"])


## 3. Fitur level eksplisit

Dihitung **dari input** encoder (bukan dari label), jadi tidak ada leakage:
`N_LEVEL_FEATS = 7` → RMS dBFS, band-low, band-mid, band-high (dB), peak dB, crest factor, active mask.

In [7]:
# ----------------------------------------------------------------------------- Cell 4: level features
N_LEVEL_FEATS = 7  # rms, band_low, band_mid, band_high, peak, crest, active


def _assemble_level(rms_db: torch.Tensor, bands: torch.Tensor, peak_db: torch.Tensor) -> torch.Tensor:
    crest = peak_db - rms_db
    active = (rms_db > ACTIVE_THRESHOLD_DB).to(rms_db.dtype)
    stacked = torch.stack(
        [rms_db, bands[..., 0], bands[..., 1], bands[..., 2], peak_db, crest, active * 30.0], dim=-1
    )
    return stacked / 30.0  # rough fixed scaling; the level MLP handles the rest


def mel_level_features(audio_norm: torch.Tensor) -> torch.Tensor:
    """Level features from normalized dBFS log-mel (B, 4, 64, T). dB = x * 50 - 30."""
    db = audio_norm * 50.0 - 30.0
    power = torch.pow(10.0, db / 10.0)
    rms_db = 10.0 * torch.log10(power.mean(dim=(2, 3)).clamp_min(1e-12))
    peak_db = db.amax(dim=(2, 3))
    bands = 10.0 * torch.log10(
        torch.stack(
            [power[:, :, :16].mean(dim=(2, 3)), power[:, :, 16:40].mean(dim=(2, 3)), power[:, :, 40:].mean(dim=(2, 3))],
            dim=-1,
        ).clamp_min(1e-12)
    )
    return _assemble_level(rms_db, bands, peak_db)


def waveform_level_features(waveform: torch.Tensor, sample_rate: int = 16000) -> torch.Tensor:
    """Level features from raw waveform (B, 4, S)."""
    x = waveform.float()
    rms_db = 20.0 * torch.log10(x.pow(2).mean(dim=-1).clamp_min(1e-12).sqrt())
    peak_db = 20.0 * torch.log10(x.abs().amax(dim=-1).clamp_min(1e-12))
    spec = torch.fft.rfft(x, dim=-1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    freqs = torch.fft.rfftfreq(x.shape[-1], d=1.0 / sample_rate).to(x.device)
    band_db = []
    for lo, hi in ((0, 250), (250, 2000), (2000, sample_rate // 2)):
        sel = (freqs >= lo) & (freqs < hi)
        band_db.append(10.0 * torch.log10(power[..., sel].mean(dim=-1).clamp_min(1e-12)))
    return _assemble_level(rms_db, torch.stack(band_db, dim=-1), peak_db)


def compute_level_features(x: torch.Tensor, input_key: str) -> torch.Tensor:
    """All encoders use the same waveform-level features.

    The encoder content path may differ, but level features are always
    computed from the 16 kHz waveform so RMS/band/peak values are directly
    comparable in dBFS.
    """
    if input_key != "waveform":
        raise ValueError(f"Level features require waveform input, got {input_key!r}.")
    return waveform_level_features(x)

## 4. Text encoder yang di-fine-tune

`TextEncoderWrapper` bawaan proyek membekukan encoder teks (`@torch.no_grad` + cache), sehingga
tidak bisa dipakai untuk fine-tune. Cabang kecil di bawah ini **meneruskan gradien** ke IndoBERT.

In [8]:
# ----------------------------------------------------------------------------- Cell 5: trainable text branch


class TrainableIndoBertTextBranch(nn.Module):
    """Fine-tunable IndoBERT producing bounded balance targets r (B, 4)."""

    is_pretrained = True

    def __init__(self, model_name: str, hidden_dim: int = 128, dropout: float = 0.15, max_length: int = 64):
        super().__init__()
        from transformers import AutoModel, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.max_length = max_length
        embed_dim = int(self.model.config.hidden_size)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4),
        )

    def forward(self, prompts, device) -> torch.Tensor:
        encoded = self.tokenizer(
            list(prompts), padding=True, truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        hidden = self.model(**encoded).last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return 6.0 * torch.tanh(self.head(pooled) / 6.0)

## 5. Audio encoder registry

Antarmuka seragam: `forward(input) -> tokens (B, 4, T', D)`.
- `cnn_scratch` → CNN 1D scratch pada waveform 16 kHz, menghasilkan 25 token temporal.
- `ast` / `mert` / `beats` → encoder pretrained dari waveform (S = 128000), dijalankan per-stem
  dengan bobot yang sama (shared), semua parameter **trainable**.
- Semua encoder memakai input waveform 16 kHz dan level features dari waveform yang sama.
- BEATs memakai AMP **OFF** karena fbank/AMP FP16 tidak stabil di T4.
- Saat training, notebook mencetak loss/gradient finite, scaler scale, dan optimizer step count.

In [9]:
# ----------------------------------------------------------------------------- Cell 6: audio encoders
BEATS_URL = "https://huggingface.co/lpepino/beats_ckpts/resolve/main/BEATs_iter3_plus_AS2M.pt"


def download_beats_checkpoint(path: Path) -> Path:
    path = Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading BEATs checkpoint -> {path}")
    from urllib.request import urlretrieve

    urlretrieve(BEATS_URL, path)
    return path


def set_encoder_dropout(encoder: nn.Module | None, p: float) -> None:
    """Set dropout probability only inside the audio encoder.

    Text-branch dropout is controlled separately by CONFIG['dropout_text'].
    """
    if encoder is None:
        return
    p = float(p)
    for module in encoder.modules():
        if isinstance(module, nn.Dropout):
            module.p = p


class ScratchWaveformEncoder(nn.Module):
    """Scratch 1D CNN on raw 16 kHz waveform, returning 25 temporal tokens per stem."""

    input_key = "waveform"
    is_pretrained = False

    def __init__(self, embed_dim: int = 128, n_tokens: int = 25):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_tokens = n_tokens
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=4, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=4, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=3, stride=4, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )
        self.proj = nn.Sequential(
            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )
        self.dropout = nn.Dropout(0.0)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, n_samples = waveform.shape
        x = waveform.reshape(bsz * n_stems, 1, n_samples)
        h = self.conv(x)
        h = F.adaptive_avg_pool1d(h, self.n_tokens)
        h = h.transpose(1, 2)
        h = self.proj(h)
        h = self.dropout(h)
        return h.reshape(bsz, n_stems, self.n_tokens, self.embed_dim)


class _HFWaveformEncoder(nn.Module):
    input_key = "waveform"
    is_pretrained = True
    sample_rate = 16000

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:  # (N, S) -> (N, T, D)
        raise NotImplementedError

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, _ = waveform.shape
        flat = waveform.reshape(bsz * n_stems, waveform.shape[-1])
        hidden = self._encode_waveform(flat)
        return hidden.reshape(bsz, n_stems, hidden.shape[-2], hidden.shape[-1])


class ASTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593"):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel

        self.extractor = ASTFeatureExtractor.from_pretrained(model_name)
        self.model = ASTModel.from_pretrained(model_name)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()), sampling_rate=self.sample_rate, return_tensors="pt"
        )
        return self.model(feats["input_values"].to(waveforms.device)).last_hidden_state


class MERTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "m-a-p/MERT-v1-95M"):
        super().__init__()
        from transformers import AutoConfig, AutoModel, Wav2Vec2FeatureExtractor

        cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        if not hasattr(cfg, "conv_pos_batch_norm"):
            cfg.conv_pos_batch_norm = False
        self.model = AutoModel.from_pretrained(model_name, config=cfg, trust_remote_code=True)
        self.extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name, trust_remote_code=True)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        # Wav2Vec2FeatureExtractor memvalidasi sampling_rate == extractor.sampling_rate (24 kHz)
        # tanpa melakukan resampling; tetap kirim waveform 16 kHz sesuai protokol eksperimen.
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()),
            sampling_rate=self.extractor.sampling_rate,
            return_tensors="pt",
        )
        outputs = self.model(feats["input_values"].to(waveforms.device))
        return outputs.last_hidden_state


BEATS_REPO_DIR = ROOT / "third_party" / "unilm"
BEATS_PACKAGE_DIR = BEATS_REPO_DIR / "beats"


def ensure_beats() -> Path:
    """Clone Microsoft unilm and expose the BEATs source directory on sys.path."""
    import shutil
    import subprocess

    if BEATS_PACKAGE_DIR.is_dir():
        if str(BEATS_PACKAGE_DIR) not in sys.path:
            sys.path.insert(0, str(BEATS_PACKAGE_DIR))
        try:
            import torchaudio  # noqa: F401
        except Exception as exc:  # pragma: no cover - dependency gate
            raise RuntimeError(
                "BEATs membutuhkan torchaudio. Install torchaudio yang cocok dengan torch Kaggle."
            ) from exc
        return BEATS_PACKAGE_DIR

    BEATS_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    if BEATS_REPO_DIR.exists() and not (BEATS_REPO_DIR / ".git").exists():
        shutil.rmtree(BEATS_REPO_DIR, ignore_errors=True)

    if not (BEATS_REPO_DIR / ".git").exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/microsoft/unilm.git",
                str(BEATS_REPO_DIR),
            ],
            check=True,
        )

    if not BEATS_PACKAGE_DIR.is_dir():
        raise RuntimeError(
            f"Folder BEATs tidak ditemukan setelah clone: {BEATS_PACKAGE_DIR}"
        )
    if str(BEATS_PACKAGE_DIR) not in sys.path:
        sys.path.insert(0, str(BEATS_PACKAGE_DIR))

    try:
        import torchaudio  # noqa: F401
    except Exception as exc:  # pragma: no cover - dependency gate
        raise RuntimeError(
            "BEATs membutuhkan torchaudio. Install torchaudio yang cocok dengan torch Kaggle."
        ) from exc

    return BEATS_PACKAGE_DIR


class BEATsEncoder(_HFWaveformEncoder):
    def __init__(self, checkpoint: str | None = None):
        super().__init__()
        try:
            from beats.BEATs import BEATs, BEATsConfig
        except Exception:
            ensure_beats()
            from BEATs import BEATs, BEATsConfig

        ckpt = download_beats_checkpoint(Path(checkpoint) if checkpoint else Path.home() / "beats_data" / "BEATs_iter3_plus_AS2M.pt")
        payload = torch.load(ckpt, map_location="cpu", weights_only=False)
        config = BEATsConfig(payload["cfg"])
        self.model = BEATs(config)
        self.model.load_state_dict(payload["model"])
        self.embed_dim = int(config.encoder_embed_dim)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        out = self.model.extract_features(waveforms)
        return out[0] if isinstance(out, tuple) else out


ENCODER_REGISTRY = {
    "cnn_scratch": ScratchWaveformEncoder,
    "ast": ASTEncoder,
    "mert": MERTEncoder,
    "beats": BEATsEncoder,
}


def build_encoder(name: str) -> nn.Module:
    if name not in ENCODER_REGISTRY:
        raise ValueError(f"Unknown encoder '{name}'. Available: {sorted(ENCODER_REGISTRY)}")
    return ENCODER_REGISTRY[name]()


## 6. `EncoderProbeModel` (audio encoder + IndoBERT + head M3)

Hanya blok audio encoder yang berganti. Head M3, rumus `g`, dan kontrak loss tetap identik.
`audio_input_key` sekarang selalu `waveform`, sehingga semua encoder dievaluasi dengan
`evaluate_model` yang sama.

In [10]:
# ----------------------------------------------------------------------------- Cell 7: model


class EncoderProbeModel(nn.Module):
    """M3 structured two-branch head with a swappable audio encoder and a fine-tuned IndoBERT branch."""

    def __init__(self, encoder_name: str, mode: str, config: dict):
        super().__init__()
        if mode not in ("bimodal", "audio_only", "text_only"):
            raise ValueError(f"Invalid mode: {mode}")
        self.mode = mode
        self.encoder_name = encoder_name
        # text_only tidak memakai audio encoder (g = r): jangan bangun/unduh encoder berat.
        self.encoder = None if mode == "text_only" else build_encoder(encoder_name)
        self.audio_input_key = self.encoder.input_key if self.encoder is not None else "audio"

        d = config["d_model"]
        self.enc_proj = nn.Linear(self.encoder.embed_dim if self.encoder is not None else d, d)
        self.stem_embedding = nn.Parameter(torch.zeros(1, 4, 1, d))
        self.level_mlp = nn.Sequential(nn.Linear(N_LEVEL_FEATS, d), nn.GELU(), nn.Linear(d, d))
        self.stem_norm = nn.LayerNorm(d)
        self.loudness_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.text_branch = TrainableIndoBertTextBranch(
            config["text_model"], config["text_hidden"], config["dropout_text"]
        )

    def forward(self, audio_input, prompts, active_mask=None, clamp_output: bool = False) -> dict:
        device = audio_input.device
        bsz = audio_input.shape[0]
        if active_mask is not None:
            mask = active_mask.to(device=device, dtype=torch.float32)
        else:
            mask = torch.ones((bsz, 4), device=device)
        count = mask.sum(dim=1, keepdim=True).clamp_min(1.0)

        l_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "text_only":
            level = compute_level_features(audio_input, self.audio_input_key)
            tokens = self.encoder(audio_input)
            h = self.enc_proj(tokens) + self.stem_embedding + self.level_mlp(level).unsqueeze(2)
            l_pred = self.loudness_head(self.stem_norm(h.mean(dim=2))).squeeze(-1)
        l_ref = (l_pred * mask).sum(dim=1, keepdim=True) / count

        r_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "audio_only":
            raw_r = self.text_branch(prompts, device)
            r_pred = (raw_r - (raw_r * mask).sum(dim=1, keepdim=True) / count) * mask

        g_pred = (l_ref + r_pred - l_pred) * mask
        if clamp_output or not self.training:
            g_pred = g_pred.clamp(-18.0, 12.0)
        return {"gains_pred": g_pred, "loudness_pred": l_pred, "balance_pred": r_pred}

    def build_param_groups(self, config: dict) -> list[dict]:
        """Separate encoder / text / head groups so tuning does not leak across branches."""
        groups: list[dict] = []
        if self.encoder is not None:
            enc_lr = config["lr_audio"] if getattr(self.encoder, "is_pretrained", False) else config["lr_scratch"]
            groups.append({
                "params": list(self.encoder.parameters()),
                "lr": enc_lr,
                "weight_decay": config["wd_audio"],
            })

        head_modules = [self.enc_proj, self.level_mlp, self.stem_norm, self.loudness_head, self.text_branch.head]
        head_params = [p for module in head_modules for p in module.parameters()] + [self.stem_embedding]
        groups.append({
            "params": list(self.text_branch.model.parameters()),
            "lr": config["lr_text"],
            "weight_decay": config["wd_text"],
        })
        groups.append({
            "params": head_params,
            "lr": config["lr_head"],
            "weight_decay": config["wd_head"],
        })
        return groups


## 7. Train / eval loop

Reuse `compute_multimodal_loss` dan `evaluate_model`. Early stopping pada val MAE, AMP, gradient
accumulation, cosine schedule. Bobot aux loss mengikuti mode ablasi.

In [11]:
# ----------------------------------------------------------------------------- Cell 8: training loop


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def env_info() -> dict:
    info = {"torch": torch.__version__, "device": str(DEVICE)}
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["cuda"] = torch.version.cuda
    return info


def run_one(
    encoder_name: str,
    mode: str,
    seed: int,
    config: dict,
    train_ds: Dataset,
    val_ds: Dataset,
    device=DEVICE,
    fold_id: int | None = None,
    params: dict | None = None,
    eval_test: bool = False,
    save_state: bool = False,
    checkpoint_path: Path | None = None,
) -> dict:
    seed_everything(seed)
    model = EncoderProbeModel(encoder_name, mode, config).to(device)
    set_encoder_dropout(model.encoder, config["dropout_audio"])
    groups = model.build_param_groups(config)
    optimizer = torch.optim.AdamW(groups, weight_decay=0.0)
    opt_step_count = 0

    def _count_opt_step(*_args, **_kwargs) -> None:
        nonlocal opt_step_count
        opt_step_count += 1

    optimizer.register_step_post_hook(_count_opt_step)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(config["epochs"], 1), eta_min=config["eta_min"]
    )
    amp = bool(
        config["use_amp"]
        and device.type == "cuda"
        and encoder_name not in AMP_DISABLED_ENCODERS
    )
    amp_dtype = AMP_DTYPE if amp else torch.float32
    use_scaler = bool(amp and amp_dtype == torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    all_params = [p for g in groups for p in g["params"]]
    print(f"  [amp] {encoder_name}: {'ON (' + str(amp_dtype).split('.')[-1] + ')' if amp else 'OFF'} | batch={config['batch_size']}x{config['grad_accum_steps']}")

    dl_kwargs = {
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY and device.type == "cuda",
    }
    if NUM_WORKERS > 0:
        dl_kwargs["persistent_workers"] = True
        dl_kwargs["prefetch_factor"] = 2

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, **dl_kwargs)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, **dl_kwargs)
    accum = max(1, int(config["grad_accum_steps"]))
    lam_l, lam_r = AUX_LAMBDA[mode]

    best = {"val_mae": float("inf"), "epoch": 0, "state": None, "metrics": {}}
    history: list[dict] = []
    t0 = time.time()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            audio = batch[model.audio_input_key].to(device=device, dtype=torch.float32, non_blocking=PIN_MEMORY)
            mask = batch["active_mask"].to(device=device, non_blocking=PIN_MEMORY)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp):
                out = model(audio, batch["prompt"], active_mask=mask, clamp_output=False)
                losses = compute_multimodal_loss(
                    out,
                    batch["gains_db"].to(device=device, dtype=torch.float32, non_blocking=PIN_MEMORY),
                    batch["loudness_db"].to(device=device, dtype=torch.float32, non_blocking=PIN_MEMORY),
                    batch["balance_target"].to(device=device, dtype=torch.float32, non_blocking=PIN_MEMORY),
                    active_mask=mask,
                    lambda_loudness=lam_l,
                    lambda_balance=lam_r,
                )
                loss = losses["loss"]

            if DEBUG_NUMERICS and step == 0:
                print(
                    f"  [debug] loss={loss.item():.6f} "
                    f"finite={bool(torch.isfinite(loss).item())} "
                    f"g_pred_finite={bool(torch.isfinite(out['gains_pred']).all().item())}"
                )
            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss pada {encoder_name}/fold{fold_id}/epoch{epoch}/step{step}"
                )

            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                if config["grad_clip"] > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(all_params, config["grad_clip"])

                first_opt_step = min(accum, len(train_loader)) - 1
                if DEBUG_NUMERICS and step == first_opt_step:
                    grads = [p.grad for p in all_params if p.grad is not None]
                    finite_grads = sum(
                        bool(torch.isfinite(g).all().item()) for g in grads
                    )
                    print(
                        f"  [debug] grads={len(grads)} finite_grads={finite_grads} "
                        f"scaler_scale={scaler.get_scale():.1f}"
                    )
                    bad = [
                        name for name, p in model.named_parameters()
                        if p.grad is not None and not torch.isfinite(p.grad).all()
                    ]
                    if bad:
                        if amp:
                            print(
                                f"  [debug] non-finite grad (AMP akan skip & backoff) pada "
                                f"{encoder_name}/fold{fold_id}: {bad[:5]}"
                            )
                        else:
                            raise RuntimeError(
                                f"Non-finite gradient pada {encoder_name}/fold{fold_id}: {bad[:5]}"
                            )

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        val = evaluate_model(model, val_loader, device=device)
        history.append({
            "epoch": epoch,
            "val_mae_db": val["mae_db"],
            "val_rmse_db": val["rmse_db"],
            "val_band_mae_db": val["band_mae_db"],
            "val_success_1db": val["success_rate_1db"],
        })
        fold_tag = f"/f{fold_id}" if fold_id is not None else ""
        print(
            f"[{encoder_name:11s}/{mode:10s}{fold_tag}/seed{seed}] ep{epoch:02d} "
            f"val_MAE={val['mae_db']:.4f} RMSE={val['rmse_db']:.4f} band={val['band_mae_db']:.4f}"
        )
        if val["mae_db"] < best["val_mae"] - 1e-4:
            state = copy.deepcopy(model.state_dict()) if save_state else None
            best = {"val_mae": val["mae_db"], "epoch": epoch, "state": state, "metrics": val}
        elif config["patience"] and (epoch - best["epoch"]) >= config["patience"]:
            print(f"  early stop at epoch {epoch} (best {best['epoch']})")
            break

    if opt_step_count == 0:
        raise RuntimeError(
            f"Optimizer tidak pernah step pada {encoder_name}/fold{fold_id}. "
            "Cek AMP, loss, dan gradient."
        )
    print(f"  [debug] optimizer_step_count={opt_step_count}")

    if save_state and best["state"] is not None:
        model.load_state_dict(best["state"])

    test_human = None
    test_synthetic = None
    if eval_test:
        if Path(config["test_human_manifest"]).is_file():
            test_loader = DataLoader(
                EncoderDataset(config["test_human_manifest"]),
                batch_size=config["batch_size"],
                shuffle=False,
                **dl_kwargs,
            )
            test_human = evaluate_model(model, test_loader, device=device)
        if Path(config["test_synthetic_manifest"]).is_file():
            test_loader = DataLoader(
                EncoderDataset(config["test_synthetic_manifest"]),
                batch_size=config["batch_size"],
                shuffle=False,
                **dl_kwargs,
            )
            test_synthetic = evaluate_model(model, test_loader, device=device)

    if checkpoint_path is not None and save_state:
        checkpoint_path = Path(checkpoint_path)
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "state_dict": model.state_dict(),
            "config": config,
            "params": params or {},
            "encoder": encoder_name,
            "mode": mode,
            "seed": seed,
            "best_epoch": best["epoch"],
            "validation": best["metrics"],
            "test_human": test_human,
            "test_synthetic": test_synthetic,
        }, checkpoint_path)

    return {
        "encoder": encoder_name,
        "mode": mode,
        "seed": seed,
        "fold": fold_id,
        "params": params or {},
        "member": MEMBER,
        "env": env_info(),
        "trainable_params": n_params,
        "train_sec": round(time.time() - t0, 1),
        "best_epoch": best["epoch"],
        "validation": best["metrics"],
        "test_human": test_human,
        "test_synthetic": test_synthetic,
        "checkpoint": str(checkpoint_path) if checkpoint_path is not None and save_state else None,
        "history": history,
    }


## 8. Holdout + OFAT (auto-run)

Cell 9 menjalankan `run_auto()` jika `AUTO_RUN=True`.

### Phase 1 — per member

- Audio tuning holdout **one-factor-at-a-time (OFAT)**.
- Order: `lr_audio`/`lr_scratch` → `wd_audio` → `dropout_audio`.
- Setiap konfigurasi: train 80 lagu, early stop di 20 validation.
- Tidak ada 5-fold CV.
- Audio-only ablation holdout × 1 seed.
- Text-only ablation hanya member 2, setelah `text_config.json` ada.

### Phase 2 — winner

- `AUTO_PICK_WINNER=True` memilih encoder dengan validation MAE terendah.
- Text tuning global **tidak otomatis**; pakai baseline `dropout_text=0.15` atau set manual.
- Member pemilik winner menjalankan final model 3 seed.
- Final dievaluasi pada `test_human` dan `test_synthetic`.

### Skip otomatis

- Audio tuning selesai → skip.
- `audio_only_summary_*.json` ada → skip.
- `text_config.json` ada → skip text tuning.
- `text_only_summary.json` ada → skip.
- `final_summary_*.json` ada → skip.

Test set tidak pernah dipakai selama tuning.


In [ ]:
# ----------------------------------------------------------------------------- Cell 9: holdout + OFAT + final
# No 5-fold CV. Tuning uses fixed 80 train / 20 validation.


def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def compact_result(result: dict) -> dict:
    out = {k: v for k, v in result.items() if k not in ("history", "validation", "test_human", "test_synthetic")}
    val = dict(result.get("validation", {}))
    val.pop("per_song", None)
    out["validation"] = val
    for key in ("test_human", "test_synthetic"):
        if result.get(key):
            m = dict(result[key])
            m.pop("per_song", None)
            out[key] = m
    return out


def make_config(params: dict) -> dict:
    cfg = copy.deepcopy(CONFIG)
    cfg.update(params or {})
    return cfg


def mean_std(values):
    arr = np.asarray(list(values), dtype=float)
    return float(arr.mean()), float(arr.std(ddof=0))


def write_rows_csv(path, rows):
    import csv

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def summarize_ablation(rows, encoder, mode, params):
    maes = [r["mae_db"] for r in rows]
    rmses = [r["rmse_db"] for r in rows]
    mae_mean, mae_std = mean_std(maes)
    rmse_mean, _ = mean_std(rmses)
    return {
        "encoder": encoder,
        "member": MEMBER,
        "mode": mode,
        "params": params,
        "n_folds": len({r["fold"] for r in rows}),
        "n_seeds": len({r["seed"] for r in rows}),
        "mae_db_mean": round(mae_mean, 4),
        "mae_db_std": round(mae_std, 4),
        "rmse_db_mean": round(rmse_mean, 4),
        "rows": rows,
    }


def _config_trials_done(path: Path) -> int:
    if not Path(path).is_file():
        return 0
    try:
        return int(json.loads(Path(path).read_text()).get("n_trials", 0))
    except Exception:
        return 0


def sync_outputs_to_drive(include_checkpoints: bool = False) -> None:
    """Copy local outputs/encoder_selection/{memberN,text,final} to Google Drive."""
    if not Path("/content/drive/MyDrive").exists():
        return
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for sub in (f"member{MEMBER}", "text"):
        src = OUT_DIR / sub
        if src.is_dir() and any(src.iterdir()):
            shutil.copytree(src, DRIVE_OUTPUT_DIR / sub, dirs_exist_ok=True)
    if FINAL_DIR.is_dir() and any(FINAL_DIR.iterdir()):
        dst_final = DRIVE_OUTPUT_DIR / "final"
        dst_final.mkdir(parents=True, exist_ok=True)
        for p in FINAL_DIR.iterdir():
            if p.is_file() and (include_checkpoints or p.suffix != ".pt"):
                shutil.copy2(p, dst_final / p.name)
    for top_name in ("audio_encoder_results.csv", "audio_encoder_mae.png"):
        top_file = OUT_DIR / top_name
        if top_file.is_file():
            shutil.copy2(top_file, DRIVE_OUTPUT_DIR / top_name)


def run_one_with_retry(encoder_name, mode, seed, config, train_ds, val_ds, **kwargs):
    import gc

    candidates = []
    bs = int(config["batch_size"])
    for cand in (bs, 8, 4, 2):
        if cand <= bs and cand not in candidates and effective_batch % cand == 0:
            candidates.append(cand)
    last_exc = None
    for idx, cand_bs in enumerate(candidates):
        cfg = copy.deepcopy(config)
        cfg["batch_size"] = cand_bs
        cfg["grad_accum_steps"] = effective_batch // cand_bs
        if idx > 0:
            print(f"OOM -> retry dengan batch_size={cand_bs}, grad_accum_steps={cfg['grad_accum_steps']} (effective batch tetap {effective_batch})")
        try:
            return run_one(encoder_name, mode, seed, cfg, train_ds, val_ds, **kwargs)
        except torch.cuda.OutOfMemoryError as exc:
            last_exc = exc
            gc.collect()
            torch.cuda.empty_cache()
    raise last_exc


_HOLDOUT_CACHE = {}
HOLDOUT_DIR = RUN_DIR / "holdout"
HOLDOUT_DIR.mkdir(parents=True, exist_ok=True)


def holdout_datasets():
    if not _HOLDOUT_CACHE:
        _HOLDOUT_CACHE["train"] = EncoderDataset(CONFIG["train_manifest"])
        _HOLDOUT_CACHE["val"] = EncoderDataset(CONFIG["val_manifest"])
    return _HOLDOUT_CACHE["train"], _HOLDOUT_CACHE["val"]


def evaluate_params_holdout(encoder_name: str, params: dict, tag: str,
                            mode: str = "bimodal", seed: int | None = None) -> dict:
    seed = int(CONFIG["tuning_seed"]) if seed is None else int(seed)
    train_ds, val_ds = holdout_datasets()
    cfg = make_config(params)
    result = run_one_with_retry(
        encoder_name,
        mode,
        seed,
        cfg,
        train_ds,
        val_ds,
        fold_id=None,
        params=params,
        eval_test=False,
        save_state=False,
    )
    save_json(HOLDOUT_DIR / f"{tag}.json", compact_result(result))
    row = {"seed": seed, "fold": 0}
    val = dict(result["validation"])
    val.pop("per_song", None)
    row.update(val)
    return row


def _base_audio_params(encoder_name: str) -> dict:
    if encoder_name == "cnn_scratch":
        return {
            "lr_scratch": CONFIG["lr_scratch"],
            "wd_audio": CONFIG["wd_audio"],
            "dropout_audio": CONFIG["dropout_audio"],
        }
    return {
        "lr_audio": CONFIG["lr_audio"],
        "wd_audio": CONFIG["wd_audio"],
        "dropout_audio": CONFIG["dropout_audio"],
    }


def run_audio_tuning():
    if "beats" in SHARDS[MEMBER]["encoders"]:
        ensure_beats()
    if "bimodal" not in SHARDS[MEMBER]["modes"]:
        print("member ini tidak punya mode bimodal; skip audio tuning")
        return

    for encoder_name in SHARDS[MEMBER]["encoders"]:
        best_path = RUN_DIR / f"best_audio_config_{encoder_name}.json"
        if best_path.is_file() and not FORCE_RERUN:
            print(f"skip audio tuning {encoder_name}: {best_path.name} sudah ada")
            continue

        base = _base_audio_params(encoder_name)
        space = AUDIO_SEARCH_SPACE["scratch" if encoder_name == "cnn_scratch" else "pretrained"]
        lr_key = "lr_scratch" if encoder_name == "cnn_scratch" else "lr_audio"
        stages = [
            (1, lr_key, space[lr_key]),
            (2, "wd_audio", space["wd_audio"]),
            (3, "dropout_audio", space["dropout_audio"]),
        ]

        best_params = dict(base)
        best_metrics = None
        rows = []
        for stage, key, values in stages:
            for value in values:
                params = dict(best_params)
                params[key] = value
                tag = f"ofat_{encoder_name}_s{stage}_{key}_{value}"
                row = evaluate_params_holdout(encoder_name, params, tag=tag)
                row.update({"stage": stage, "knob": key, "value": value})
                rows.append(row)
                print(
                    f"[ofat] {encoder_name} stage{stage} {key}={value} "
                    f"val_MAE={row['mae_db']:.4f}"
                )
                if best_metrics is None or row["mae_db"] < best_metrics["mae_db"]:
                    best_metrics = row
                    best_params = params

        payload = {
            "encoder": encoder_name,
            "member": MEMBER,
            "mode": "bimodal",
            "params": best_params,
            "val_mae": round(float(best_metrics["mae_db"]), 4),
            "val_rmse": round(float(best_metrics["rmse_db"]), 4),
            "cv_mae_mean": round(float(best_metrics["mae_db"]), 4),
            "cv_mae_std": 0.0,
            "cv_rmse_mean": round(float(best_metrics["rmse_db"]), 4),
            "n_folds": 1,
            "tuning_seed": int(CONFIG["tuning_seed"]),
            "n_trials": len(rows),
            "fold_metrics": [best_metrics],
        }
        save_json(best_path, payload)
        write_rows_csv(RUN_DIR / f"ofat_summary_{encoder_name}.csv", rows)
        print(f"[audio] {encoder_name}: val MAE {payload['val_mae']:.4f} | params={best_params}")


def load_best_audio(encoder_name: str) -> dict:
    candidates = [RUN_DIR / f"best_audio_config_{encoder_name}.json"]
    candidates.extend(sorted(OUT_DIR.glob(f"member*/best_audio_config_{encoder_name}.json")))
    for path in candidates:
        if Path(path).is_file():
            return json.loads(Path(path).read_text())
    raise FileNotFoundError(f"best_audio_config_{encoder_name}.json belum ada")


def _base_text_params() -> dict:
    return {
        "lr_text": CONFIG["lr_text"],
        "dropout_text": CONFIG["dropout_text"],
        "text_hidden": CONFIG["text_hidden"],
    }


def run_text_tuning():
    if TEXT_TUNING_ENCODER is None or TEXT_TUNING_MEMBER is None:
        print("TEXT_TUNING_ENCODER / TEXT_TUNING_MEMBER belum diset; skip text tuning")
        return
    if int(MEMBER) != int(TEXT_TUNING_MEMBER):
        print(f"text tuning hanya dijalankan oleh member {TEXT_TUNING_MEMBER}; member ini {MEMBER}")
        return
    if TEXT_TUNING_ENCODER not in SHARDS[MEMBER]["encoders"]:
        print(f"{TEXT_TUNING_ENCODER} bukan encoder member {MEMBER}; skip")
        return

    text_config_path = TEXT_DIR / "text_config.json"
    if text_config_path.is_file() and not FORCE_RERUN:
        print(f"skip text tuning: {text_config_path.name} sudah ada")
        return

    best_audio = load_best_audio(TEXT_TUNING_ENCODER)
    audio_params = best_audio["params"]
    stages = [
        (1, "lr_text", TEXT_SEARCH_SPACE["lr_text"]),
        (2, "dropout_text", TEXT_SEARCH_SPACE["dropout_text"]),
        (3, "text_hidden", TEXT_SEARCH_SPACE["text_hidden"]),
    ]

    best_text = _base_text_params()
    best_metrics = None
    rows = []
    for stage, key, values in stages:
        for value in values:
            text_params = dict(best_text)
            text_params[key] = value
            params = {**audio_params, **text_params}
            tag = f"ofat_text_s{stage}_{key}_{value}"
            row = evaluate_params_holdout(TEXT_TUNING_ENCODER, params, tag=tag)
            row.update({"stage": stage, "knob": key, "value": value})
            rows.append(row)
            print(f"[ofat-text] stage{stage} {key}={value} val_MAE={row['mae_db']:.4f}")
            if best_metrics is None or row["mae_db"] < best_metrics["mae_db"]:
                best_metrics = row
                best_text = text_params

    payload = {
        "encoder": TEXT_TUNING_ENCODER,
        "member": MEMBER,
        "params": best_text,
        "audio_params": audio_params,
        "val_mae": round(float(best_metrics["mae_db"]), 4),
        "cv_mae_mean": round(float(best_metrics["mae_db"]), 4),
        "cv_mae_std": 0.0,
        "n_folds": 1,
        "tuning_seed": int(CONFIG["tuning_seed"]),
        "n_trials": len(rows),
        "fold_metrics": [best_metrics],
    }
    save_json(text_config_path, payload)
    write_rows_csv(TEXT_DIR / "ofat_summary_text.csv", rows)
    print(f"[text] {TEXT_TUNING_ENCODER}: val MAE {payload['val_mae']:.4f} | params={best_text}")


def run_audio_only():
    if "beats" in SHARDS[MEMBER]["encoders"]:
        ensure_beats()
    if "audio_only" not in SHARDS[MEMBER]["modes"]:
        print(f"audio_only bukan mode member {MEMBER}; skip")
        return
    for encoder_name in SHARDS[MEMBER]["encoders"]:
        summary_path = RUN_DIR / f"audio_only_summary_{encoder_name}.json"
        if summary_path.is_file() and not FORCE_RERUN:
            print(f"skip audio_only {encoder_name}: {summary_path.name} sudah ada")
            continue
        try:
            best_audio = load_best_audio(encoder_name)
        except FileNotFoundError as exc:
            print(f"audio_only {encoder_name}: skip ({exc})")
            continue
        params = best_audio["params"]
        rows = []
        for seed in CONFIG.get("audio_only_seeds", [CONFIG["tuning_seed"]]):
            row = evaluate_params_holdout(
                encoder_name, params, tag=f"audio_only_{encoder_name}_seed{seed}",
                mode="audio_only", seed=int(seed),
            )
            rows.append(row)
        summary = summarize_ablation(rows, encoder_name, "audio_only", params)
        save_json(summary_path, summary)
        print(f"[audio_only] {encoder_name}: MAE {summary['mae_db_mean']:.4f}")


def run_text_only():
    if "text_only" not in SHARDS[MEMBER]["modes"]:
        print(f"text_only bukan mode member {MEMBER}; skip")
        return
    summary_path = RUN_DIR / "text_only_summary.json"
    if summary_path.is_file() and not FORCE_RERUN:
        print(f"skip text_only: {summary_path.name} sudah ada")
        return
    text_path = TEXT_DIR / "text_config.json"
    if text_path.is_file():
        params = json.loads(text_path.read_text())["params"]
    else:
        params = _base_text_params()
    rows = []
    for seed in CONFIG["seeds"]:
        row = evaluate_params_holdout(
            "ast", params, tag=f"text_only_seed{seed}", mode="text_only", seed=int(seed)
        )
        rows.append(row)
    summary = summarize_ablation(rows, "none", "text_only", params)
    save_json(summary_path, summary)
    print(f"[text_only] MAE {summary['mae_db_mean']:.4f}")


def run_final():
    if FINAL_ENCODER is None:
        print("FINAL_ENCODER belum diset; skip final")
        return
    if FINAL_ENCODER not in SHARDS[MEMBER]["encoders"]:
        print(f"{FINAL_ENCODER} bukan encoder member {MEMBER}; skip final")
        return
    if FINAL_ENCODER == "beats":
        ensure_beats()
    final_summary_path = FINAL_DIR / f"final_summary_{FINAL_ENCODER}.json"
    if final_summary_path.is_file() and not FORCE_RERUN:
        print(f"skip final: {final_summary_path.name} sudah ada")
        return

    best_audio = load_best_audio(FINAL_ENCODER)
    text_path = TEXT_DIR / "text_config.json"
    if text_path.is_file():
        text_params = json.loads(text_path.read_text())["params"]
    else:
        text_params = _base_text_params()
    params = {**best_audio["params"], **text_params}
    cfg = make_config(params)

    train_ds, val_ds = holdout_datasets()
    results = []
    for seed in CONFIG["seeds"]:
        ckpt = FINAL_DIR / f"best_model_{FINAL_ENCODER}_seed{seed}.pt" if SAVE_FINAL_CHECKPOINT else None
        result = run_one_with_retry(
            FINAL_ENCODER,
            "bimodal",
            int(seed),
            cfg,
            train_ds,
            val_ds,
            params=params,
            eval_test=True,
            save_state=True,
            checkpoint_path=ckpt,
        )
        save_json(FINAL_DIR / f"final_{FINAL_ENCODER}_seed{seed}.json", result)
        results.append(result)

    summary = {
        "encoder": FINAL_ENCODER,
        "member": MEMBER,
        "params": params,
        "n_seeds": len(results),
        "seeds": list(CONFIG["seeds"]),
        "test_human": {},
        "test_synthetic": {},
    }
    for key in ("test_human", "test_synthetic"):
        vals = [r[key]["mae_db"] for r in results if r.get(key)]
        rmses = [r[key]["rmse_db"] for r in results if r.get(key)]
        if vals:
            mae_mean, mae_std = mean_std(vals)
            rmse_mean, _ = mean_std(rmses)
            summary[key] = {
                "mae_db_mean": round(mae_mean, 4),
                "mae_db_std": round(mae_std, 4),
                "rmse_db_mean": round(rmse_mean, 4),
            }
    save_json(final_summary_path, summary)
    print(f"[final] {FINAL_ENCODER}: {summary['test_human']} | {summary['test_synthetic']}")


def find_winner():
    rows = []
    for path in sorted(OUT_DIR.glob("member*/best_audio_config_*.json")):
        try:
            payload = json.loads(path.read_text())
        except Exception:
            continue
        score = payload.get("val_mae", payload.get("cv_mae_mean"))
        if score is None:
            continue
        rows.append(payload)
    by_encoder = {row["encoder"]: row for row in rows}
    if set(by_encoder) != set(CONFIG["encoders"]):
        return None
    return min(by_encoder.values(), key=lambda row: float(row.get("val_mae", row.get("cv_mae_mean"))))


def run_auto():
    global FINAL_ENCODER

    print(f"AUTO_RUN member {MEMBER} | holdout+OFAT | pick_winner={AUTO_PICK_WINNER} | force={FORCE_RERUN}")

    if "beats" in SHARDS[MEMBER]["encoders"]:
        ensure_beats()

    winner = find_winner() if AUTO_PICK_WINNER else None
    if winner is not None:
        if FINAL_ENCODER is None:
            FINAL_ENCODER = winner["encoder"]
        print(
            f"winner: {winner['encoder']} (member {winner['member']}, "
            f"val MAE {winner.get('val_mae', winner.get('cv_mae_mean')):.4f})"
        )
    elif AUTO_PICK_WINNER:
        print("winner belum bisa ditentukan: belum semua best_audio_config_*.json ada")

    if "bimodal" in SHARDS[MEMBER]["modes"]:
        need_audio = any(
            not (RUN_DIR / f"best_audio_config_{enc}.json").is_file()
            for enc in SHARDS[MEMBER]["encoders"]
        )
        if need_audio or FORCE_RERUN:
            run_audio_tuning()
        else:
            print("skip audio tuning: semua best_audio_config sudah ada")

    if "audio_only" in SHARDS[MEMBER]["modes"]:
        for enc in SHARDS[MEMBER]["encoders"]:
            if not (RUN_DIR / f"best_audio_config_{enc}.json").is_file():
                print(f"skip audio_only {enc}: best_audio_config belum ada")
                continue
            if (RUN_DIR / f"audio_only_summary_{enc}.json").is_file() and not FORCE_RERUN:
                print(f"skip audio_only {enc}: summary sudah ada")
                continue
            run_audio_only()
            break

    text_cfg_path = TEXT_DIR / "text_config.json"
    if TEXT_TUNING_ENCODER and TEXT_TUNING_MEMBER and int(MEMBER) == int(TEXT_TUNING_MEMBER):
        if text_cfg_path.is_file() and not FORCE_RERUN:
            print("skip text tuning: text_config.json sudah ada")
        else:
            run_text_tuning()
    else:
        print("skip text tuning: tidak diset manual (pakai text_config baseline)")

    text_tuning_done = text_cfg_path.is_file()
    if "text_only" in SHARDS[MEMBER]["modes"]:
        if not text_tuning_done:
            print("skip text_only: text_config.json belum ada")
        elif (RUN_DIR / "text_only_summary.json").is_file() and not FORCE_RERUN:
            print("skip text_only: text_only_summary.json sudah ada")
        else:
            run_text_only()

    if FINAL_ENCODER and FINAL_ENCODER in SHARDS[MEMBER]["encoders"]:
        if not text_tuning_done:
            print("skip final: text_config.json belum ada")
        elif (FINAL_DIR / f"final_summary_{FINAL_ENCODER}.json").is_file() and not FORCE_RERUN:
            print(f"skip final: final_summary_{FINAL_ENCODER}.json sudah ada")
        else:
            run_final()
    else:
        print("skip final: member ini bukan owner FINAL_ENCODER / FINAL_ENCODER belum diset")

    sync_outputs_to_drive(include_checkpoints=True)
    print("AUTO_RUN selesai & tersinkron ke Drive:", DRIVE_OUTPUT_DIR)


if AUTO_RUN:
    run_auto()
else:
    if RUN_AUDIO_TUNING:
        run_audio_tuning()
    if RUN_TEXT_TUNING:
        run_text_tuning()
    if RUN_FINAL:
        run_final()
    if RUN_AUDIO_ONLY:
        run_audio_only()
    if RUN_TEXT_ONLY:
        run_text_only()
    if not any([RUN_AUDIO_TUNING, RUN_TEXT_TUNING, RUN_FINAL, RUN_AUDIO_ONLY, RUN_TEXT_ONLY]):
        print("AUTO_RUN=False dan tidak ada manual flag yang aktif.")


## 9. Leaderboard

Leaderboard menggabungkan:
- `best_audio_config_<encoder>.json` → CV MAE mean ± std.
- `final/final_summary_<encoder>.json` → test MAE mean ± std.

Pemenang dipilih dari **CV MAE terendah**, lalu dikonfirmasi pada test set.


In [13]:
# ----------------------------------------------------------------------------- Cell 10: leaderboard (CV + final)
import csv


def load_json(path):
    return json.loads(Path(path).read_text())


def load_audio_configs():
    rows = []
    for path in sorted(OUT_DIR.glob("member*/best_audio_config_*.json")):
        try:
            d = load_json(path)
        except Exception as exc:  # noqa: BLE001
            print("skip", path, exc)
            continue
        rows.append(d)
    return rows


def load_final_summaries():
    out = {}
    for path in sorted(OUT_DIR.glob("final/final_summary_*.json")):
        try:
            d = load_json(path)
        except Exception as exc:  # noqa: BLE001
            print("skip", path, exc)
            continue
        out[d["encoder"]] = d
    return out


AUDIO_ROWS = load_audio_configs()
FINAL_ROWS = load_final_summaries()

TEXT_CFG = {}
_text_path = TEXT_DIR / "text_config.json"
if _text_path.is_file():
    TEXT_CFG = json.loads(_text_path.read_text()).get("params", {})

SUMMARY = []
for row in AUDIO_ROWS:
    encoder = row["encoder"]
    final = FINAL_ROWS.get(encoder, {})
    th = final.get("test_human", {})
    ts = final.get("test_synthetic", {})
    SUMMARY.append({
        "encoder": encoder,
        "member": row.get("member"),
        "cv_mae_mean": row.get("cv_mae_mean"),
        "cv_mae_std": row.get("cv_mae_std"),
        "cv_rmse_mean": row.get("cv_rmse_mean"),
        "test_human_mae_mean": th.get("mae_db_mean"),
        "test_human_mae_std": th.get("mae_db_std"),
        "test_synthetic_mae_mean": ts.get("mae_db_mean"),
        "test_synthetic_mae_std": ts.get("mae_db_std"),
        "n_seeds": final.get("n_seeds"),
        "params": json.dumps(row.get("params", {}), sort_keys=True),
        "text_params": json.dumps(TEXT_CFG, sort_keys=True),
    })

SUMMARY = sorted(SUMMARY, key=lambda r: (r["cv_mae_mean"] is None, float("inf") if r["cv_mae_mean"] is None else r["cv_mae_mean"]))
if SUMMARY:
    csv_path = OUT_DIR / "audio_encoder_results.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(SUMMARY[0].keys()))
        writer.writeheader()
        writer.writerows(SUMMARY)
    print("wrote", csv_path)
    print(f"{'encoder':12s} {'member':>6s} {'CV MAE':>16s} {'test_human':>16s} {'test_synth':>16s}")
    for r in SUMMARY:
        cv = f"{r['cv_mae_mean']:.4f}±{r['cv_mae_std']:.4f}" if r["cv_mae_mean"] is not None else "-"
        th = f"{r['test_human_mae_mean']:.4f}" if r["test_human_mae_mean"] is not None else "-"
        ts = f"{r['test_synthetic_mae_mean']:.4f}" if r["test_synthetic_mae_mean"] is not None else "-"
        print(f"{r['encoder']:12s} {str(r['member']):>6s} {cv:>16s} {th:>16s} {ts:>16s}")
    print("\nBest by CV MAE:", SUMMARY[0]["encoder"], SUMMARY[0]["cv_mae_mean"])
else:
    print("Belum ada best_audio_config_*.json. Jalankan audio tuning dulu.")


wrote /content/wadidaw-studio/outputs/encoder_selection/audio_encoder_results.csv
encoder      member           CV MAE       test_human       test_synth
cnn_scratch       1    0.0888±0.0180                -                -
beats             4    1.4000±1.0041                -                -

Best by CV MAE: cnn_scratch 0.0888


In [14]:
# ----------------------------------------------------------------------------- Cell 11: figure (CV MAE by encoder)
if SUMMARY:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    labels = [r["encoder"] for r in SUMMARY]
    vals = [r["cv_mae_mean"] if r["cv_mae_mean"] is not None else 0.0 for r in SUMMARY]
    errs = [r["cv_mae_std"] if r["cv_mae_std"] is not None else 0.0 for r in SUMMARY]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(labels, vals, yerr=errs, capsize=3)
    ax.set_ylabel("5-fold CV MAE (dB)")
    ax.set_title("Audio encoder comparison (Optuna, 5-fold CV)")
    fig.tight_layout()
    fig_path = OUT_DIR / "audio_encoder_mae.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.close(fig)
else:
    print("nothing to plot yet")


wrote /content/wadidaw-studio/outputs/encoder_selection/audio_encoder_mae.png


## 10. Parity / sanity checks

Memastikan:
- Holdout tetap: 80 train / 20 validation / 50 test.
- Tidak ada song overlap antar-split.
- Tidak ada prompt overlap antar-split.
- Tidak ada 5-fold CV.
- Semua parameter trainable.
- Text config tunggal.
- Shard encoder menutup semua kombinasi.


In [ ]:
# ----------------------------------------------------------------------------- Cell 12: parity checks
import collections

def _manifest_records(path):
    data = json.loads(Path(path).read_text())
    return data["samples"] if isinstance(data, dict) and "samples" in data else data

FIXED_KEYS = [
    "text_model", "d_model", "text_hidden", "dropout_text", "dropout_audio",
    "batch_size", "grad_accum_steps", "epochs", "patience",
    "lr_head", "lr_audio", "lr_scratch", "lr_text",
    "wd_audio", "wd_text", "wd_head", "eta_min", "grad_clip", "seeds",
    "tuning_seed",
]
for key in FIXED_KEYS:
    assert key in CONFIG, f"missing fixed knob: {key}"
assert set(CONFIG["modes"]) == {"bimodal", "audio_only", "text_only"}
assert CONFIG["batch_size"] * CONFIG["grad_accum_steps"] == effective_batch, "effective batch harus 16"
assert MEMBER in SHARDS

# Shard coverage.
coverage = collections.defaultdict(set)
for member, shard in SHARDS.items():
    for enc in shard["encoders"]:
        for mode in shard["modes"]:
            coverage[(enc, mode)].add(member)
expected = {(e, m) for e in CONFIG["encoders"] for m in ("bimodal", "audio_only")}
missing = expected - set(coverage)
assert not missing, f"shard tidak menutup: {sorted(missing)}"
text_only_members = [m for m, s in SHARDS.items() if "text_only" in s["modes"]]
assert len(text_only_members) == 1, f"text_only harus tepat satu anggota, bukan {text_only_members}"

# Holdout checks: fixed train/validation/test, no 5-fold CV.
train = _manifest_records(CONFIG["train_manifest"])
val = _manifest_records(CONFIG["val_manifest"])
test_h = _manifest_records(CONFIG["test_human_manifest"])
test_s = _manifest_records(CONFIG["test_synthetic_manifest"])
train_songs = {str(r["song"]) for r in train}
val_songs = {str(r["song"]) for r in val}
test_songs = {str(r["song"]) for r in test_h} | {str(r["song"]) for r in test_s}
assert len(train_songs) == 80, f"train songs harus 80, bukan {len(train_songs)}"
assert len(val_songs) == 20, f"val songs harus 20, bukan {len(val_songs)}"
assert not (train_songs & val_songs), "train/validation song overlap"
assert not (train_songs & test_songs), "train/test song overlap"
assert not (val_songs & test_songs), "validation/test song overlap"
assert len(train) == 3200 and len(val) == 800, "holdout sample counts tidak sesuai"
assert len(test_h) == 200 and len(test_s) == 2000, "test sample counts tidak sesuai"

train_prompts = {str(r["prompt"]) for r in train}
val_prompts = {str(r["prompt"]) for r in val}
test_prompts = {str(r["prompt"]) for r in test_h} | {str(r["prompt"]) for r in test_s}
assert not (train_prompts & val_prompts), "train/validation prompt overlap"
assert not (train_prompts & test_prompts), "train/test prompt overlap"
assert not (val_prompts & test_prompts), "validation/test prompt overlap"

# Nothing frozen.
probe = EncoderProbeModel("cnn_scratch", "bimodal", CONFIG)
frozen = [name for name, p in probe.named_parameters() if not p.requires_grad]
assert not frozen, f"frozen params found: {frozen}"
assert all(p.requires_grad for p in probe.text_branch.model.parameters())
assert EncoderProbeModel("ast", "text_only", CONFIG).encoder is None

# Text config, if present, must be a single global file.
text_cfg = TEXT_DIR / "text_config.json"
if text_cfg.is_file():
    payload = json.loads(text_cfg.read_text())
    assert set(payload["params"]) == set(TEXT_SEARCH_SPACE), f"text params mismatch: {payload['params']}"

print(f"parity OK | holdout 80/20/50 | no 5-fold CV | text_only member {text_only_members[0]}")
if text_cfg.is_file():
    print("text config:", json.loads(text_cfg.read_text())["params"])
print("parity checks passed.")


In [16]:
# ----------------------------------------------------------------------------- Cell 13: package outputs (Colab + Google Drive)
import shutil

sync_outputs_to_drive(include_checkpoints=True)

member_dir = OUT_DIR / f"member{MEMBER}"
member_files = [p for p in sorted(member_dir.glob("**/*")) if p.is_file()] if member_dir.is_dir() else []
extra_files = [
    p
    for d in (TEXT_DIR, FINAL_DIR)
    if d.is_dir()
    for p in sorted(d.glob("**/*"))
    if p.is_file()
]
result_files = member_files + extra_files
if result_files:
    local_base = Path("/content" if Path("/content").exists() else ".") / f"encoder_selection_member{MEMBER}"
    archive = Path(shutil.make_archive(str(local_base), "zip", root_dir=OUT_DIR))
    print("wrote", archive, f"({archive.stat().st_size / 1e6:.2f} MB)")
    if Path("/content/drive/MyDrive").exists():
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        drive_zip = DRIVE_OUTPUT_DIR / archive.name
        shutil.copy2(archive, drive_zip)
        print("copied zip -> Drive:", drive_zip)
    for path in result_files:
        print(" -", path.relative_to(OUT_DIR))
    print(f"\nHasil tersimpan di {DRIVE_OUTPUT_DIR}. Commit JSON/CSV di outputs/encoder_selection/member{MEMBER}/ ke repo.")
else:
    print("belum ada hasil untuk di-zip; jalankan stage tuning terlebih dahulu")


wrote /content/encoder_selection_member4.zip (1.33 MB)
copied zip -> Drive: /content/drive/MyDrive/ml_dabd_nlp/ml/2.Dokumentasi/m4/encoder_selection_member4.zip
 - member4/audio_only_summary_beats.json
 - member4/best_audio_config_beats.json
 - member4/cv/audio_beats_trial000_fold0.json
 - member4/cv/audio_beats_trial000_fold1.json
 - member4/cv/audio_beats_trial000_fold2.json
 - member4/cv/audio_beats_trial000_fold3.json
 - member4/cv/audio_beats_trial000_fold4.json
 - member4/cv/audio_only_beats_seed42_fold0.json
 - member4/cv/audio_only_beats_seed42_fold1.json
 - member4/cv/audio_only_beats_seed42_fold2.json
 - member4/cv/audio_only_beats_seed42_fold3.json
 - member4/cv/audio_only_beats_seed42_fold4.json
 - member4/cv_summary_audio_only_beats.csv
 - member4/cv_summary_beats.csv
 - member4/optuna/audio_beats.db

Hasil tersimpan di /content/drive/MyDrive/ml_dabd_nlp/ml/2.Dokumentasi/m4. Commit JSON/CSV di outputs/encoder_selection/member4/ ke repo.


## 11. Kesimpulan & handoff ke step 2

1. Pilih encoder pemenang dari **validation MAE** (holdout 20 lagu).
2. Text tuning OFAT opsional; baseline `dropout_text=0.15` tetap valid.
3. Latih final model 3 seed dan laporkan **test MAE mean ± std**.
4. Bawa encoder + best audio config + best text config ke **step 2** untuk sweep fusion.

**Keterbatasan yang harus ditulis:**
- MERT diberi input 16 kHz (aslinya 24 kHz) — domain shift.
- Pretraining dataset AST/MERT/BEATs berbeda — perbandingan pretraining tidak netral.
- Jumlah parameter encoder berbeda; yang disamakan adalah protokol.
- Validation score setelah tuning bersifat optimistis; test set adalah estimasi final.


## Lanjut setelah sesi Colab

1. Cek folder Google Drive `ml_dabd_nlp/ml/2.Dokumentasi/m_all/` — pastikan `member1/` … `member4/`, `text/`, dan `final/` sudah masuk.
2. Putuskan runtime Colab (**Runtime → Disconnect and delete runtime**) agar CCU tidak terpakai saat idle.
3. Salin isi folder member ke `outputs/encoder_selection/member<N>/` di repo lokal, lalu `git add`, `commit`, dan `push`.
4. Setelah semua member push, jalankan `git pull` di member pemenang, lalu jalankan lagi notebook (`AUTO_PICK_WINNER=True`) untuk final 3 seed.
5. Member 2 menjalankan notebook lagi setelah `text_config.json` tersedia untuk `text_only` ablation.

**Catatan teknis (Colab L4 / A100)**
- Cache `logmel` (593 MB) dan `waveform16k` (1,5 GB) dibaca dari SSD lokal `/content/wadidaw-studio/data/cache/` untuk menghindari bottleneck FUSE Drive, lalu `waveform16k` otomatis di-backup ke `1.Dataset/cache/waveform16k`.
- Pada L4 (24 GB) dan A100 (40/80 GB), notebook mengaktifkan TF32 + `bfloat16` AMP serta menyesuaikan physical `batch_size` (`8×2` di L4, `16×1` di A100) sambil mengunci `effective_batch = 16`.
- `MEMBER=4` meng-clone `microsoft/unilm` ke `third_party/unilm` dan menambahkan `beats/` ke `sys.path`.
- MERT memakai `trust_remote_code=True` dan patch `conv_pos_batch_norm=False` untuk kompatibilitas `transformers` terbaru.
- Tidak ada Optuna/5-fold; tuning memakai OFAT holdout dan hasil per-run disimpan ke `member<N>/holdout/`.
